# Uber Eats Marketplace Lakehouse - Demo Quickstart

This notebook demonstrates the complete medallion architecture (Bronze → Silver → Gold) for the Uber Eats marketplace data platform.

**Architecture Overview:**
* 🥉 **Bronze Layer**: Raw ingested data (12 tables)
* 🥈 **Silver Layer**: Cleaned & transformed data (12 tables)
* 🥇 **Gold Layer**: Analytics-ready star schema (19 tables - facts, dimensions, aggregates, views)

In [0]:
%sql
-- Show the complete catalog structure
SHOW SCHEMAS IN ue_marketplace_lakehouse_dev

## 🥉 Bronze Layer - Raw Data
Raw data ingested from source systems without transformation

In [0]:
%sql
-- View all bronze tables
SHOW TABLES IN ue_marketplace_lakehouse_dev.bronze

In [0]:
%sql
-- Sample raw orders data
SELECT * FROM ue_marketplace_lakehouse_dev.bronze.orders_raw 
LIMIT 10

## 🥈 Silver Layer - Cleaned & Transformed
Data quality rules applied, standardized formats, business logic implemented

In [0]:
%sql
-- View all silver tables
SHOW TABLES IN ue_marketplace_lakehouse_dev.silver

In [0]:
%sql
-- Sample cleaned orders with enhanced data quality
SELECT 
  order_id,
  customer_id,
  merchant_id,
  courier_id,
  order_status,
  order_total,
  delivery_fee,
  tax_amount,
  order_timestamp,
  delivery_timestamp
FROM ue_marketplace_lakehouse_dev.silver.orders 
LIMIT 10

## 🥇 Gold Layer - Analytics-Ready Star Schema
Optimized for BI and analytics with fact tables, dimensions, and pre-aggregated metrics

In [0]:
%sql
-- View all gold layer objects
SHOW TABLES IN ue_marketplace_lakehouse_dev.gold

## 📊 Business Analytics - Key Metrics

In [0]:
%sql
-- Marketplace Executive Summary
SELECT * FROM ue_marketplace_lakehouse_dev.gold.vw_marketplace_executive_summary
LIMIT 20

In [0]:
%sql
-- Daily order volume and revenue trends
SELECT 
  f.order_date,
  DATE_FORMAT(f.order_date, 'EEEE') as day_name,
  COUNT(DISTINCT f.order_id) as total_orders,
  SUM(f.total_amount) as total_revenue,
  AVG(f.total_amount) as avg_order_value,
  SUM(f.delivery_fee) as total_delivery_fees,
  COUNT(DISTINCT f.customer_id) as unique_customers,
  COUNT(DISTINCT f.merchant_id) as active_merchants
FROM ue_marketplace_lakehouse_dev.gold.fact_order f
WHERE f.order_date >= CURRENT_DATE - INTERVAL 30 DAYS
GROUP BY f.order_date
ORDER BY f.order_date DESC
LIMIT 30

In [0]:
%sql
-- Top 10 merchants by revenue
SELECT 
  m.merchant_name,
  m.city_id,
  m.cuisine_type,
  COUNT(DISTINCT f.order_id) as total_orders,
  SUM(f.total_amount) as total_revenue,
  AVG(f.total_amount) as avg_order_value,
  AVG(f.actual_delivery_minutes) as avg_delivery_time_mins
FROM ue_marketplace_lakehouse_dev.gold.fact_order f
JOIN ue_marketplace_lakehouse_dev.gold.dim_merchant m ON f.merchant_sk = m.merchant_sk
WHERE f.order_date >= CURRENT_DATE - INTERVAL 30 DAYS
GROUP BY m.merchant_name, m.city_id, m.cuisine_type
ORDER BY total_revenue DESC
LIMIT 10

In [0]:
%sql
-- Delivery SLA analysis
SELECT * FROM ue_marketplace_lakehouse_dev.gold.vw_delivery_sla_analysis
ORDER BY delivery_date DESC
LIMIT 20

In [0]:
%sql
-- City-wise hourly marketplace health metrics
SELECT 
  city_name,
  date_hour,
  total_orders,
  total_revenue,
  avg_order_value,
  active_merchants,
  active_couriers,
  avg_delivery_time_minutes,
  on_time_delivery_rate
FROM ue_marketplace_lakehouse_dev.gold.agg_city_hourly_marketplace_health
WHERE date_hour >= CURRENT_TIMESTAMP - INTERVAL 7 DAYS
ORDER BY date_hour DESC, total_revenue DESC
LIMIT 50

In [0]:
%sql
-- Customer ordering patterns
SELECT 
  c.loyalty_tier,
  COUNT(DISTINCT c.customer_id) as total_customers,
  COUNT(DISTINCT f.order_id) as total_orders,
  SUM(f.total_amount) as total_spent,
  AVG(f.total_amount) as avg_order_value,
  COUNT(DISTINCT f.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) as orders_per_customer
FROM ue_marketplace_lakehouse_dev.gold.fact_order f
JOIN ue_marketplace_lakehouse_dev.gold.dim_customer c ON f.customer_sk = c.customer_sk
GROUP BY c.loyalty_tier
ORDER BY total_spent DESC

In [0]:
%sql
-- Payment methods distribution
SELECT 
  payment_method,
  payment_status,
  COUNT(*) as transaction_count,
  SUM(total_amount) as total_revenue,
  AVG(total_amount) as avg_transaction_value
FROM ue_marketplace_lakehouse_dev.gold.fact_payment
GROUP BY payment_method, payment_status
ORDER BY total_revenue DESC

In [0]:
# Python analysis: Revenue trends visualization
import pandas as pd

# Query daily revenue trends
df = spark.sql("""
  SELECT 
    f.order_date,
    SUM(f.total_amount) as daily_revenue,
    COUNT(DISTINCT f.order_id) as daily_orders,
    AVG(f.total_amount) as avg_order_value
  FROM ue_marketplace_lakehouse_dev.gold.fact_order f
  WHERE f.order_date >= CURRENT_DATE - INTERVAL 30 DAYS
  GROUP BY f.order_date
  ORDER BY f.order_date
""").toPandas()

print(f"Total records: {len(df)}")
print(f"\nRevenue Summary:")
print(f"Total Revenue: ${df['daily_revenue'].sum():,.2f}")
print(f"Average Daily Revenue: ${df['daily_revenue'].mean():,.2f}")
print(f"Total Orders: {df['daily_orders'].sum():,.0f}")
print(f"Average Daily Orders: {df['daily_orders'].mean():,.1f}")

display(df)

## 🎯 Demo Summary

This Uber Eats Marketplace Lakehouse demonstrates:

1. **Medallion Architecture**: Bronze → Silver → Gold data layers
2. **Data Quality**: Automated validation and cleansing in silver layer
3. **Star Schema Design**: Optimized dimensional model for analytics
4. **Business Metrics**: Pre-aggregated KPIs for fast reporting
5. **Multi-Language Support**: SQL and Python for flexibility

**Key Features:**
* 📦 12 Bronze tables (raw data)
* ✨ 12 Silver tables (cleaned data)
* 💎 19 Gold objects (7 facts, 7 dimensions, 2 aggregates, 3 views)
* 🚀 Ready for BI dashboards and ML workflows